# EMA Pullback Strategy Notebook

This notebook runs the EMA Pullback strategy and provides detailed performance analysis including MFE/MAE, Veto reasons, and contextual filtering (VIX/Chop).

In [ ]:
import os
import sys
from dataclasses import asdict
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

def find_repo_root(start: Path) -> Path:
    target = Path("scripts/trading_framework/config/sessions.yaml")
    for p in [start, *start.parents]:
        if (p / target).exists():
            return p
    raise FileNotFoundError("Could not locate repo root containing sessions.yaml")

ROOT = find_repo_root(Path.cwd().resolve())
os.chdir(ROOT)
CONFIG_PATH = ROOT / "scripts" / "trading_framework" / "config" / "sessions.yaml"

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from scripts.trading_framework.config.config_loader import load_config
from scripts.libs_py.data.loader import DataLoader
from scripts.libs_py.features.feature_registry import FeatureRegistry
from scripts.strategies.ema_pullback.core.ema_pullback import EMAPullbackStrategy
from scripts.trading_framework.core.mfe_mae import compute_mfe_mae_rich, summarize_mfe_mae_rich
from scripts.trading_framework.core.signal_adapter import enrich_signals, split_approved_vetoed

SYMBOL = "NQ"
MAX_SIGNALS = 100
MAX_FORWARD_BARS = 120
FEATURES_NEEDED = [
    "atr_14",
    "vix_regime",
    "chop_score",
    "chop_regime",
    "ema_9",
    "ema_20",
    "ema_50",
    "ema_200",
]

print("Repo root:", ROOT)
print("CWD:", Path.cwd())
print("Config path:", CONFIG_PATH)
print("Notebook symbol:", SYMBOL)
print("Max signals per bucket:", MAX_SIGNALS)

In [ ]:
cfg = load_config(str(CONFIG_PATH))
point_value = cfg.execution.point_value.get(SYMBOL, 20.0)

loader = DataLoader(cfg)
df = loader.load_enriched(SYMBOL)
registry = FeatureRegistry(cfg)
df = registry.ensure_features(df, FEATURES_NEEDED)

strategy = EMAPullbackStrategy(ticker=SYMBOL)
raw_signals = strategy.hunt(df)
enriched = enrich_signals(
    raw_signals,
    df,
    strategy_name="ema_pullback",
    symbol=SYMBOL,
    point_value=point_value,
)
approved, vetoed = split_approved_vetoed(enriched)

approved_run = approved.head(MAX_SIGNALS).copy()
vetoed_run = vetoed.head(MAX_SIGNALS).copy()

approved_results = compute_mfe_mae_rich(
    df,
    approved_run,
    max_forward_bars=MAX_FORWARD_BARS,
    horizons=cfg.mfe_mae.forward_horizons_minutes,
    atr_col="atr_14",
)
summary_approved = summarize_mfe_mae_rich(approved_results)

def summary_table(summary: dict) -> pd.DataFrame:
    if not summary:
        return pd.DataFrame()
    rows = [
        {"metric": "mfe_points", "p25": summary.get("mfe_p25"), "p50": summary.get("mfe_p50"), "p75": summary.get("mfe_p75")},
        {"metric": "mae_points", "p25": summary.get("mae_p25"), "p50": summary.get("mae_p50"), "p75": summary.get("mae_p75")},
        {"metric": "mfe_atr", "p25": summary.get("mfe_atr_p25"), "p50": summary.get("mfe_atr_p50"), "p75": summary.get("mfe_atr_p75")},
        {"metric": "mae_atr", "p25": summary.get("mae_atr_p25"), "p50": summary.get("mae_atr_p50"), "p75": summary.get("mae_atr_p75")},
    ]
    return pd.DataFrame(rows).round(4)

def conditional_tables(frame: pd.DataFrame, results: list) -> dict[str, pd.DataFrame]:
    if frame.empty or not results:
        return {}
    merged = frame.head(len(results)).copy()
    merged["peak_mfe"] = [r.mfe_points[-1] if r.mfe_points else 0.0 for r in results]
    merged["peak_mae"] = [r.mae_points[-1] if r.mae_points else 0.0 for r in results]
    merged["reached_1r"] = [r.reached_1r for r in results]
    merged["reached_2r"] = [r.reached_2r for r in results]

    tables = {}
    for group_col in ["context_session_block", "context_vix_regime", "context_chop_regime", "direction"]:
        if group_col not in merged.columns:
            continue
        grouped = merged.groupby(group_col, dropna=False).agg(
            count=("peak_mfe", "size"),
            mfe_median=("peak_mfe", "median"),
            mae_median=("peak_mae", "median"),
            pct_reach_1r=("reached_1r", "mean"),
            pct_reach_2r=("reached_2r", "mean"),
        ).round(4)
        tables[group_col] = grouped
    return tables

veto_counts = vetoed["veto_reason"].value_counts().rename_axis("veto_reason").reset_index(name="count")
key_metrics = pd.DataFrame([
    {
        "bucket": "approved",
        "count": len(approved),
        "pct_reach_1r": summary_approved.get("pct_reach_1r"),
        "pct_reach_2r": summary_approved.get("pct_reach_2r"),
        "median_mfe_atr": summary_approved.get("mfe_atr_p50"),
        "median_mae_atr": summary_approved.get("mae_atr_p50"),
    }
]).round(4)

display(Markdown(f"## Analysis Results for {SYMBOL}"))
display(key_metrics)

display(Markdown("### Top Veto Reasons"))
display(veto_counts.head(10))

display(Markdown("### Approved Summary"))
display(summary_table(summary_approved))

approved_tables = conditional_tables(approved_run, approved_results)
for group_col, table in approved_tables.items():
    display(Markdown(f"### Conditional Table: {group_col}"))
    display(table)